In [ ]:
import os, time
from pathlib import Path
import torch
import psutil
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display, Markdown

%matplotlib inline

display(Markdown("## Part 0 — Setup"))
display(Markdown("_Imports, helpers, RAM tracker, and the model ID. Nothing is downloaded or loaded yet._"))

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["font.size"] = 10

CATEGORY_COLORS = {
    "weights":          "#d62728",
    "tokenizer":        "#2ca02c",
    "model-config":     "#1f77b4",
    "metadata/license": "#9467bd",
    "other":            "#7f7f7f",
}

PROC = psutil.Process(os.getpid())
def ram_mb(): return PROC.memory_info().rss / (1024 ** 2)

def fmt_size(b):
    if b < 1024:        return f"{b} B"
    if b < 1024**2:     return f"{b/1024:.2f} KB"
    if b < 1024**3:     return f"{b/1024**2:.2f} MB"
    return f"{b/1024**3:.2f} GB"

STAGE_LOG = []

class Stage:
    """Reusable timing + RAM tracker. Use in every level."""
    def __init__(self, name, reads=""):
        self.name, self.reads = name, reads
    def __enter__(self):
        print(f"▶ {self.name}")
        if self.reads: print(f"  reads: {self.reads}")
        self.ram_start = ram_mb()
        self.t_start = time.time()
        return self
    def __exit__(self, *a):
        dt = time.time() - self.t_start
        ram_end = ram_mb()
        delta = ram_end - self.ram_start
        STAGE_LOG.append({
            "stage":         self.name,
            "reads":         self.reads,
            "ram_before_MB": round(self.ram_start, 1),
            "ram_after_MB":  round(ram_end, 1),
            "ram_delta_MB":  round(delta, 1),
            "time_s":        round(dt, 2),
        })
        print(f"  ✓ {dt:.2f}s   ΔRAM +{delta:.1f} MB   total {ram_end:.1f} MB\n")

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@                PART 00: SETUP COMPLETE                   @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("========== ENVIRONMENT ==========")
print("model id   :", MODEL_ID)
print("torch ver  :", torch.__version__)
print("cuda avail :", torch.cuda.is_available())
print("baseline RAM (notebook only) :", f"{ram_mb():.1f} MB")
print("=================================")

In [ ]:
display(Markdown("## Part 1 — Download stage"))
display(Markdown("_Where do files land on disk? What is fetched and in what order?_"))

from huggingface_hub import snapshot_download, scan_cache_dir

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@               PART 01: DOWNLOAD STAGE                    @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

def cache_info(model_id):
    try:
        for r in scan_cache_dir().repos:
            if r.repo_id == model_id:
                return True, r.size_on_disk
    except Exception: pass
    return False, 0

cached_before, size_before = cache_info(MODEL_ID)
display(Markdown(f"**Cache before:** hit=`{cached_before}`" +
                 (f", size=`{fmt_size(size_before)}`" if cached_before else "")))

print("========== STEP 1: CACHE CHECK ==========")
print("cache hit :", cached_before)
if cached_before:
    print("cached size :", fmt_size(size_before))
print("=========================================")

print()
print("========== STEP 2: SNAPSHOT DOWNLOAD ==========")
with Stage("snapshot_download", reads="huggingface hub"):
    local_path = snapshot_download(repo_id=MODEL_ID)
print("local path :", local_path)
print("===============================================")

hf_root = Path(os.path.expanduser("~/.cache/huggingface/hub"))
display(Markdown(f"""
**Cache layout**
- root: `{hf_root}`
- snapshot: `{local_path}`
- relative: `{Path(local_path).relative_to(hf_root)}`
"""))

def categorize(name):
    n = name.lower()
    if n.endswith((".safetensors", ".bin")):                return "weights"
    if n in ("config.json", "generation_config.json"):      return "model-config"
    if "tokenizer" in n or n == "special_tokens_map.json":  return "tokenizer"
    if n.endswith((".md", ".txt")):                         return "metadata/license"
    return "other"

rows = []
for f in os.listdir(local_path):
    full = os.path.join(local_path, f)
    if os.path.isfile(full) or os.path.islink(full):
        s = os.path.getsize(full)
        rows.append({"filename": f, "category": categorize(f),
                     "size_bytes": s, "size": fmt_size(s)})

df_files = pd.DataFrame(rows).sort_values("size_bytes").reset_index(drop=True)
df_files.index = range(1, len(df_files) + 1)
df_files.index.name = "#"

display(Markdown("**Files on disk (small → large = fetch order)**"))
display(df_files[["filename", "category", "size"]])

print()
print("========== STEP 3: FILES ON DISK ==========")
print(f"file count : {len(df_files)}")
print(f"total size : {fmt_size(df_files['size_bytes'].sum())}")
for _, row in df_files.iterrows():
    print(f"  {row['category']:18s} {row['size']:>12s}  {row['filename']}")
print("===========================================")

# horizontal bar — file sizes (log scale)
fig, ax = plt.subplots(figsize=(10, max(3, len(df_files) * 0.4)))
ax.barh(df_files["filename"], df_files["size_bytes"],
        color=[CATEGORY_COLORS[c] for c in df_files["category"]])
ax.set_xscale("log")
ax.set_xlabel("size (bytes, log scale)")
ax.set_title("File sizes on disk")
ax.invert_yaxis()
cats_present = df_files["category"].unique()
ax.legend(handles=[Patch(facecolor=CATEGORY_COLORS[c], label=c) for c in cats_present],
          loc="lower right")
plt.tight_layout(); plt.show()

# # pie — disk usage by category
# cat_sizes = df_files.groupby("category")["size_bytes"].sum().sort_values(ascending=False)
# fig, ax = plt.subplots(figsize=(6, 6))
# ax.pie(cat_sizes.values, labels=cat_sizes.index,
#        colors=[CATEGORY_COLORS[c] for c in cat_sizes.index],
#        autopct=lambda p: f"{p:.1f}%\n({fmt_size(p * cat_sizes.sum() / 100)})",
#        startangle=90)
# ax.set_title("Disk usage by category"); plt.tight_layout(); plt.show()

total = df_files["size_bytes"].sum()
display(Markdown(f"""
**Summary** — files: `{len(df_files)}`, total: `{fmt_size(total)}`, fresh: `{not cached_before}`
"""))

In [ ]:
display(Markdown("## Part 2 — Load into RAM stage"))
display(Markdown("_Disk → RAM. What is read, in what order, how much memory is consumed?_"))

from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@              PART 02: LOAD INTO RAM STAGE                @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

ram_baseline = ram_mb()
display(Markdown(f"**Baseline RAM (before model load):** `{ram_baseline:.1f} MB`"))

print("========== STEP 1: BASELINE RAM ==========")
print(f"RAM before any load : {ram_baseline:.1f} MB")
print("==========================================")

log_start = len(STAGE_LOG)

print()
print("========== STEP 2: LOAD TOKENIZER ==========")
print("reads : tokenizer.json, tokenizer_config.json, special_tokens_map.json")
with Stage("load tokenizer", reads="tokenizer.json, tokenizer_config.json, special_tokens_map.json"):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("tokenizer class :", type(tokenizer).__name__)
print("vocab size      :", tokenizer.vocab_size)
print("============================================")

print()
print("========== STEP 3: LOAD MODEL CONFIG ==========")
print("reads : config.json")
with Stage("load model config", reads="config.json"):
    config = AutoConfig.from_pretrained(MODEL_ID)
print("config class :", type(config).__name__)
print("architecture :", config.architectures[0] if config.architectures else "n/a")
print("===============================================")

print()
print("========== STEP 4: LOAD MODEL WEIGHTS ==========")
print("reads : *.safetensors")
with Stage("load model weights", reads="*.safetensors"):
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    model.eval()
print("model class : ", type(model).__name__)
print("model dtype : ", next(model.parameters()).dtype)
print("model device: ", next(model.parameters()).device)
print("================================================")

df_load = pd.DataFrame(STAGE_LOG[log_start:])
display(Markdown("**Step-by-step load log**"))
display(df_load)

# RAM growth bar
labels  = ["baseline"] + df_load["stage"].tolist()
values  = [ram_baseline] + df_load["ram_after_MB"].tolist()
deltas  = [0] + df_load["ram_delta_MB"].tolist()
colors  = ["#7f7f7f", "#2ca02c", "#1f77b4", "#d62728"]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, values, color=colors)
for bar, d in zip(bars, deltas):
    h = bar.get_height()
    txt = f"{h:.0f} MB" + (f"\n(+{d:.0f})" if d > 0 else "")
    ax.text(bar.get_x() + bar.get_width()/2, h, txt, ha="center", va="bottom", fontsize=9)
ax.set_ylabel("RAM (MB)"); ax.set_title("RAM growth across load steps")
ax.set_ylim(0, max(values) * 1.15)
plt.tight_layout(); plt.show()

# time per step
fig, ax = plt.subplots(figsize=(10, 3))
ax.barh(df_load["stage"], df_load["time_s"], color=["#2ca02c", "#1f77b4", "#d62728"])
for i, v in enumerate(df_load["time_s"]):
    ax.text(v, i, f"  {v:.2f} s", va="center")
ax.set_xlabel("seconds"); ax.set_title("Time per load step")
plt.tight_layout(); plt.show()

total_params = sum(p.numel() for p in model.parameters())
param_bytes  = sum(p.numel() * p.element_size() for p in model.parameters())

print()
print("========== STEP 5: MODEL SUMMARY ==========")
print(f"architecture     : {config.architectures[0] if config.architectures else 'n/a'}")
print(f"hidden size      : {config.hidden_size}")
print(f"num layers       : {config.num_hidden_layers}")
print(f"vocab size       : {tokenizer.vocab_size}")
print(f"dtype            : {next(model.parameters()).dtype}")
print(f"device           : {next(model.parameters()).device}")
print(f"total parameters : {total_params:,}  ({total_params / 1e9:.3f} B)")
print(f"theoretical size : {fmt_size(param_bytes)}")
print(f"RAM by model     : {(ram_mb() - ram_baseline):.1f} MB")
print("===========================================")

display(Markdown(f"""
**Model summary**

| property | value |
|---|---|
| architecture     | `{config.architectures[0] if config.architectures else 'n/a'}` |
| hidden size      | `{config.hidden_size}` |
| num layers       | `{config.num_hidden_layers}` |
| vocab size       | `{tokenizer.vocab_size}` |
| dtype            | `{next(model.parameters()).dtype}` |
| device           | `{next(model.parameters()).device}` |
| total parameters | `{total_params:,}` ({total_params / 1e9:.3f} B) |
| theoretical size | `{fmt_size(param_bytes)}` |
| RAM by model     | `{(ram_mb() - ram_baseline):.1f} MB` |
"""))

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import torch

# ─────────────────────────────────────────────────────────────
# PART 3 — INFERENCE (Level 6: split `to token` into LM head + pick)
# STEP 1 — tokenize  (unchanged from Level 5)
# ─────────────────────────────────────────────────────────────

display(Markdown("## Part 3 — Inference (Level 6: split `to token` into LM head + pick)"))
display(Markdown("### Step 1 — tokenize (BPE segment + ID map)"))
display(Markdown("_Unchanged from Level 5. Text → integer token IDs via `{vocab}`._"))

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@        PART 03 / STEP 1: tokenize — Level 6              @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

# ─── IN ──────────────────────────────────────────────────────
text = "The capital of France is"

print("========== IN ==========")
print("text         :", repr(text))
print("  type       :", type(text).__name__)
print("  len (chars):", len(text))
print("========================")
print()

# ─── PROCESS (transparent) ───────────────────────────────────
vocab_size_tok = tokenizer.vocab_size
vocab_size_cfg = config.vocab_size

print("========== PROCESS (transparent) ==========")
print("step                 : tokenizer(text, return_tensors='pt')")
print("op                   : BPE segment + ID map  (string -> sub-word pieces -> integer IDs)")
print("static used          : {vocab}")
print("  tokenizer class    :", type(tokenizer).__name__)
print("  vocab_size (tok)   :", vocab_size_tok)
print("  vocab_size (cfg)   :", vocab_size_cfg)
print("  fast tokenizer     :", tokenizer.is_fast)
print("===========================================")
print()

# ─── run tokenize ────────────────────────────────────────────
inputs         = tokenizer(text, return_tensors="pt")
input_ids      = inputs.input_ids
attention_mask = inputs.attention_mask
token_ids_1d   = input_ids[0]
N_tok          = token_ids_1d.numel()

raw_tokens = tokenizer.convert_ids_to_tokens(token_ids_1d.tolist())
print("========== INTERNAL TOKENIZATION (for inspection) ==========")
print(f"N_tok = {N_tok}")
df_tokens = pd.DataFrame({
    "position": list(range(N_tok)),
    "token_id": token_ids_1d.tolist(),
    "raw_bpe":  [repr(t) for t in raw_tokens],
    "decoded":  [repr(tokenizer.decode([i])) for i in token_ids_1d.tolist()],
})
print(df_tokens.to_string(index=False))
print("============================================================")
print()

# ─── OUT ──────────────────────────────────────────────────────
print("========== OUT ==========")
print("token_ids    :", token_ids_1d.tolist())
print("  shape      :", tuple(token_ids_1d.shape), " <- [N_tok], 1D vector")
print("  dtype      :", token_ids_1d.dtype)
print("  ndim       :", token_ids_1d.ndim)
print("=========================")

# ─── Rich view ───────────────────────────────────────────────
display(Markdown("---"))
display(Markdown("### Rich view — Step 1"))

df_step1 = pd.DataFrame([
    {"side": "IN",      "name": "text",                              "value": repr(text),
     "shape": "—", "dtype": "str", "ndim": "—", "dim": "string"},
    {"side": "PROCESS", "name": "tokenizer (BPE segment + ID map)",  "value": f"{type(tokenizer).__name__}, vocab_size={vocab_size_tok}",
     "shape": "—", "dtype": "—",   "ndim": "—", "dim": "lookup (no math)"},
    {"side": "OUT",     "name": "token_ids",                         "value": str(token_ids_1d.tolist()),
     "shape": str(tuple(token_ids_1d.shape)), "dtype": str(token_ids_1d.dtype),
     "ndim": token_ids_1d.ndim, "dim": "1D vector [N_tok]"},
]).set_index("side")
display(df_step1)

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import torch

# ─────────────────────────────────────────────────────────────
# PART 3 — INFERENCE (Level 6)
# STEP 2 — embed (row lookup in {E})  — unchanged from Level 5
# ─────────────────────────────────────────────────────────────

display(Markdown("### Step 2 — embed (row lookup in `{E : vocab_size × d}`)"))
display(Markdown("_Unchanged from Level 5. `vectors = E[token_ids]`. Indexing, not matmul._"))

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@        PART 03 / STEP 2: embed (E exposed) — Level 6     @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

# ─── IN ──────────────────────────────────────────────────────
print("========== IN ==========")
print("token_ids    :", token_ids_1d.tolist())
print("  shape      :", tuple(token_ids_1d.shape))
print("  dtype      :", token_ids_1d.dtype)
print("========================")
print()

# ─── PROCESS (transparent — {E} exposed) ─────────────────────
E          = model.model.embed_tokens.weight     # [vocab_size, d]
vocab_rows = E.shape[0]
d          = E.shape[1]
E_dtype    = E.dtype
E_device   = E.device
E_bytes    = E.numel() * E.element_size()

print("========== PROCESS (transparent — {E} exposed) ==========")
print("step             : vectors = E[token_ids]")
print("op               : row lookup in {E}  (indexing, NOT matmul)")
print("static used      : {E : vocab_size × d}")
print("  E shape        :", tuple(E.shape))
print("  vocab_size     :", vocab_rows)
print("  d              :", d)
print("  dtype          :", E_dtype)
print("  device         :", E_device)
print("  memory         :", fmt_size(E_bytes))
print("==========================================================")
print()

# ─── run embed (explicit indexing) ───────────────────────────
token_ids_dev = token_ids_1d.to(E_device)
with torch.no_grad():
    vectors = E[token_ids_dev]                   # [N_tok, d]

with torch.no_grad():
    vectors_via_module = model.model.embed_tokens(token_ids_dev.unsqueeze(0))[0]
assert torch.equal(vectors, vectors_via_module), "E[token_ids] must match embed_tokens output"
print("sanity check       : E[token_ids] == embed_tokens(token_ids)  ✓")
print()

# ─── OUT ──────────────────────────────────────────────────────
print("========== OUT ==========")
print("vectors           : <tensor>")
print("  shape           :", tuple(vectors.shape), " <- [N_tok, d], 2D matrix")
print("  dtype           :", vectors.dtype)
print(f"  value summary   : min={vectors.min().item():+.4f}  "
      f"max={vectors.max().item():+.4f}  "
      f"mean={vectors.mean().item():+.4f}  "
      f"std={vectors.std().item():.4f}")
print("=========================")

# ─── Rich view ───────────────────────────────────────────────
display(Markdown("---"))
display(Markdown("### Rich view — Step 2"))

df_step2 = pd.DataFrame([
    {"side": "IN",      "name": "token_ids",                "value": str(token_ids_1d.tolist()),
     "shape": str(tuple(token_ids_1d.shape)), "dtype": str(token_ids_1d.dtype),
     "ndim": token_ids_1d.ndim, "dim": "1D vector [N_tok]"},
    {"side": "PROCESS", "name": "E[token_ids]  (row lookup in {E})",
     "value": f"E shape={tuple(E.shape)}, dtype={E_dtype}, mem={fmt_size(E_bytes)}",
     "shape": str(tuple(E.shape)), "dtype": str(E_dtype),
     "ndim": E.ndim, "dim": "indexing (no math)"},
    {"side": "OUT",     "name": "vectors",                  "value": f"min={vectors.min().item():+.3f} max={vectors.max().item():+.3f}",
     "shape": str(tuple(vectors.shape)), "dtype": str(vectors.dtype),
     "ndim": vectors.ndim, "dim": "2D matrix [N_tok × d]"},
]).set_index("side")
display(df_step2)

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import torch

# ─────────────────────────────────────────────────────────────
# PART 3 — INFERENCE (Level 6)
# STEP 3 — LLM core (opaque, vectors → vectors)  — unchanged from Level 5
# ─────────────────────────────────────────────────────────────

display(Markdown("### Step 3 — LLM core (opaque, outputs vectors)"))
display(Markdown("_Unchanged from Level 5. Returns last hidden state `[N_tok × d]`._"))

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@   PART 03 / STEP 3: LLM core (opaque, vectors out) — L6  @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

# ─── IN ──────────────────────────────────────────────────────
print("========== IN ==========")
print("vectors           : <tensor>")
print("  shape           :", tuple(vectors.shape), " <- [N_tok, d]")
print("  dtype           :", vectors.dtype)
print("========================")
print()

# ─── PROCESS (opaque) ────────────────────────────────────────
H            = model.lm_head.weight                  # [vocab_size, d] (Step 4)
final_norm   = model.model.norm
final_norm_w = final_norm.weight                     # still inside "to-token" until Level 11

embed_params      = E.numel()
head_params       = H.numel()
final_norm_params = final_norm_w.numel()
total_params      = sum(p.numel() for p in model.parameters())
core_params       = total_params - embed_params - head_params - final_norm_params
core_bytes        = (
    sum(p.numel() * p.element_size() for p in model.parameters())
    - E.numel() * E.element_size()
    - H.numel() * H.element_size()
    - final_norm_w.numel() * final_norm_w.element_size()
)
model_dtype  = next(model.parameters()).dtype
model_device = next(model.parameters()).device

print("========== PROCESS (opaque) ==========")
print("step             : model.model(inputs_embeds=vectors, ...) -> last_hidden_state")
print("op               : opaque  (N transformer blocks; head/norm/pick live downstream)")
print("static used      : {core internals}  (transformer blocks only)")
print("  total params   : {:,}".format(total_params))
print("  embed params   : {:,}  (excluded, Step 2)".format(embed_params))
print("  head params    : {:,}  (excluded, Step 4)".format(head_params))
print("  final-norm     : {:,}  (excluded, still inside Step 4 head box at L6)".format(final_norm_params))
print("  core params    : {:,}  ({:.3f} B)".format(core_params, core_params/1e9))
print("  core bytes     :", fmt_size(core_bytes))
print("======================================")
print()

# ─── run the opaque core (vectors → vectors) ────────────────
with torch.no_grad():
    core_out = model.model(
        inputs_embeds=vectors.unsqueeze(0),
        attention_mask=attention_mask.to(model_device),
        use_cache=False,
    )
hidden_states = core_out.last_hidden_state[0]        # [N_tok, d]

# ─── OUT ──────────────────────────────────────────────────────
print("========== OUT ==========")
print("vectors_out       : <tensor>")
print("  shape           :", tuple(hidden_states.shape), " <- [N_tok, d]")
print("  dtype           :", hidden_states.dtype)
print(f"  value summary   : min={hidden_states.min().item():+.4f}  "
      f"max={hidden_states.max().item():+.4f}  "
      f"mean={hidden_states.mean().item():+.4f}  "
      f"std={hidden_states.std().item():.4f}")
print("=========================")

# ─── Rich view ───────────────────────────────────────────────
display(Markdown("---"))
display(Markdown("### Rich view — Step 3"))

df_step3 = pd.DataFrame([
    {"side": "IN",      "name": "vectors",                        "value": f"min={vectors.min().item():+.3f} max={vectors.max().item():+.3f}",
     "shape": str(tuple(vectors.shape)), "dtype": str(vectors.dtype),
     "ndim": vectors.ndim, "dim": "2D matrix [N_tok × d]"},
    {"side": "PROCESS", "name": "model.model core (opaque)",      "value": f"core_params={core_params:,}, dtype={model_dtype}",
     "shape": "—", "dtype": "—", "ndim": "—", "dim": "opaque"},
    {"side": "OUT",     "name": "vectors_out (last hidden state)","value": f"min={hidden_states.min().item():+.3f} max={hidden_states.max().item():+.3f}",
     "shape": str(tuple(hidden_states.shape)), "dtype": str(hidden_states.dtype),
     "ndim": hidden_states.ndim, "dim": "2D matrix [N_tok × d]"},
]).set_index("side")
display(df_step3)

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import torch

# ─────────────────────────────────────────────────────────────
# PART 3 — INFERENCE (Level 6)
# STEP 4 — LM head (NEW)
#
# Difference vs Level 5 (`to token` was one opaque box):
#   - The first sub-step of `to token` is now exposed as its own step.
#   - Op is a linear projection: logits = vectors · {H}
#       vectors : [N_tok, d]
#       H       : [d, vocab_size]   (here we use H.T from PyTorch's [vocab_size, d])
#       logits  : [N_tok, vocab_size]
#   - {H} is named but NOT yet inspected as a matrix — its shape/values
#     are revealed at Level 7. At Level 6 we treat {head internals} as
#     "a matrix exists, used in a matmul", same opacity convention used
#     for embed at Level 3.
#
# IMPORTANT NUANCE:
#   - We pass `hidden_states` (the LlamaModel output, already post-final-norm)
#     into the LM head. Final RMSNorm is still bundled with the LM head
#     conceptually at Level 6; it gets peeled out as its own step at L11.
# ─────────────────────────────────────────────────────────────

display(Markdown("### Step 4 — LM head (matmul `vectors · {H}`)"))
display(Markdown("_New step. Linear projection from `[N_tok × d]` to `[N_tok × vocab_size]`. `{H}` is named but still opaque (opens at Level 7). Final RMSNorm is still implicitly bundled here; it peels out at Level 11._"))

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@        PART 03 / STEP 4: LM head (opaque) — Level 6      @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

# ─── IN ──────────────────────────────────────────────────────
print("========== IN ==========")
print("vectors_out       : <tensor>")
print("  shape           :", tuple(hidden_states.shape), " <- [N_tok, d]")
print("  dtype           :", hidden_states.dtype)
print("========================")
print()

# ─── PROCESS (opaque — only metadata of {H}) ─────────────────
head_module  = model.lm_head
head_dtype   = head_module.weight.dtype
head_device  = head_module.weight.device
head_bytes   = head_module.weight.numel() * head_module.weight.element_size()
vocab_size_H = head_module.weight.shape[0]            # rows of stored weight = vocab_size

print("========== PROCESS (opaque) ==========")
print("step             : logits = model.lm_head(vectors_out)")
print("op               : matmul  (vectors · {H})")
print("static used      : {head internals}  -- the {H} matrix (shape/values open at Level 7)")
print("  module class   :", type(head_module).__name__)
print("  has bias       :", head_module.bias is not None,
      "  <- LM head in Llama has no bias (pure linear projection)")
print("  H_rows         :", vocab_size_H, "  <- vocab_size")
print("  H_cols         :", head_module.weight.shape[1], "  <- d  (PyTorch stores W as [out, in] = [vocab_size, d])")
print("  dtype          :", head_dtype)
print("  device         :", head_device)
print("  memory         :", fmt_size(head_bytes))
print("note             : at Level 6 we DO NOT inspect H values.")
print("                   Op shape:  [N_tok, d] · [d, vocab_size] -> [N_tok, vocab_size]")
print("                   nn.Linear performs  vectors · H.T  internally.")
print("======================================")
print()

# ─── run LM head ─────────────────────────────────────────────
with torch.no_grad():
    logits = head_module(hidden_states)               # [N_tok, vocab_size]

# ─── OUT ──────────────────────────────────────────────────────
print("========== OUT ==========")
print("logits            : <tensor>")
print("  shape           :", tuple(logits.shape), " <- [N_tok, vocab_size], 2D matrix")
print("  dtype           :", logits.dtype)
print("  ndim            :", logits.ndim)
print("  numel           :", logits.numel())
print(f"  value summary   : min={logits.min().item():+.4f}  "
      f"max={logits.max().item():+.4f}  "
      f"mean={logits.mean().item():+.4f}  "
      f"std={logits.std().item():.4f}")
print("note             : raw, unnormalized scores. Not probabilities yet (softmax in Level 9).")
print("=========================")

# ─── Rich view ───────────────────────────────────────────────
display(Markdown("---"))
display(Markdown("### Rich view — Step 4"))

df_step4 = pd.DataFrame([
    {"side": "IN",      "name": "vectors_out",            "value": f"min={hidden_states.min().item():+.3f} max={hidden_states.max().item():+.3f}",
     "shape": str(tuple(hidden_states.shape)), "dtype": str(hidden_states.dtype),
     "ndim": hidden_states.ndim, "dim": "2D matrix [N_tok × d]"},
    {"side": "PROCESS", "name": "lm_head (matmul vectors · {H}, opaque)",
     "value": f"weight stored as [vocab_size={vocab_size_H}, d], dtype={head_dtype}, mem={fmt_size(head_bytes)}",
     "shape": str(tuple(head_module.weight.shape)), "dtype": str(head_dtype),
     "ndim": head_module.weight.ndim, "dim": "matmul"},
    {"side": "OUT",     "name": "logits",                 "value": f"min={logits.min().item():+.3f} max={logits.max().item():+.3f}",
     "shape": str(tuple(logits.shape)), "dtype": str(logits.dtype),
     "ndim": logits.ndim, "dim": "2D matrix [N_tok × vocab_size]"},
]).set_index("side")
display(df_step4)

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import torch

# ─────────────────────────────────────────────────────────────
# PART 3 — INFERENCE (Level 6)
# STEP 5 — pick (NEW; stateless)
#
# This new step ends the pipeline (before decode). At Level 6 it bundles,
# without exposing them as separate steps yet:
#   - take last row  (slice [-1])   -- peeled out at Level 8
#   - softmax         (over vocab)  -- peeled out at Level 9
#   - argmax / pick                 -- becomes the only op at Level 10
#
# Mathematically softmax is unnecessary for argmax (softmax is monotonic),
# so the result is identical whether we argmax over logits or probabilities.
# We use logits directly here; softmax becomes visible at Level 9.
# ─────────────────────────────────────────────────────────────

display(Markdown("### Step 5 — pick (argmax / sample)"))
display(Markdown("_New step. Stateless — no learned weights. At Level 6 this bundles take-last-row + softmax + argmax. Levels 8–10 peel them out one at a time._"))

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@        PART 03 / STEP 5: pick — Level 6                  @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

# ─── IN ──────────────────────────────────────────────────────
print("========== IN ==========")
print("logits            : <tensor>")
print("  shape           :", tuple(logits.shape), " <- [N_tok, vocab_size]")
print("  dtype           :", logits.dtype)
print("========================")
print()

# ─── PROCESS (transparent) ───────────────────────────────────
print("========== PROCESS (transparent) ==========")
print("step             : argmax over the last position's logits")
print("op               : (take last row) + (softmax) + (argmax)   -- bundled at Level 6")
print("static used      : —  (stateless, no learned parameters)")
print("note             : softmax is monotonic, so argmax(logits) == argmax(softmax(logits)).")
print("                   take-last-row peels out at Level 8.")
print("                   softmax peels out at Level 9.")
print("                   argmax becomes the only op at Level 10.")
print("============================================")
print()

# ─── run pick (logits -> 1 token ID) ─────────────────────────
with torch.no_grad():
    last_logits         = logits[-1]                          # [vocab_size]
    new_token_id_tensor = torch.argmax(last_logits)           # scalar
new_token_id_int = int(new_token_id_tensor.item())

# also surface the top-5 candidates for inspection (does not change the OUT)
top5_vals, top5_idx = torch.topk(last_logits.float(), k=5)
df_top5 = pd.DataFrame({
    "rank":     list(range(1, 6)),
    "token_id": top5_idx.tolist(),
    "logit":    [f"{v:+.4f}" for v in top5_vals.tolist()],
    "decoded":  [repr(tokenizer.decode([i])) for i in top5_idx.tolist()],
})
print("========== TOP-5 CANDIDATES (inspection only) ==========")
print(df_top5.to_string(index=False))
print("========================================================")
print()

# ─── OUT ──────────────────────────────────────────────────────
print("========== OUT ==========")
print("new_token_id (tensor) :", new_token_id_tensor)
print("  shape               :", tuple(new_token_id_tensor.shape), " <- scalar (0-d)")
print("  dtype               :", new_token_id_tensor.dtype)
print("new_token_id (int)    :", new_token_id_int)
print("=========================")

# ─── Rich view ───────────────────────────────────────────────
display(Markdown("---"))
display(Markdown("### Rich view — Step 5"))

df_step5 = pd.DataFrame([
    {"side": "IN",      "name": "logits",                          "value": f"min={logits.min().item():+.3f} max={logits.max().item():+.3f}",
     "shape": str(tuple(logits.shape)), "dtype": str(logits.dtype),
     "ndim": logits.ndim, "dim": "2D matrix [N_tok × vocab_size]"},
    {"side": "PROCESS", "name": "pick (take-last + softmax + argmax)",
     "value": "stateless",
     "shape": "—", "dtype": "—", "ndim": "—", "dim": "pure math"},
    {"side": "OUT",     "name": "new_token_id",                    "value": str(new_token_id_int),
     "shape": str(tuple(new_token_id_tensor.shape)), "dtype": str(new_token_id_tensor.dtype),
     "ndim": new_token_id_tensor.ndim, "dim": "scalar (0-D)"},
]).set_index("side")
display(df_step5)

In [ ]:
from IPython.display import display, Markdown
import pandas as pd

# ─────────────────────────────────────────────────────────────
# PART 3 — INFERENCE (Level 6)
# STEP 6 — decode (inverse vocab lookup)
#
# Unchanged op vs Level 5. Step number shifted 5 → 6 because the
# `to token` opaque box was split into LM head (Step 4) + pick (Step 5).
# ─────────────────────────────────────────────────────────────

display(Markdown("### Step 6 — decode (inverse vocab lookup)"))
display(Markdown("_Unchanged from Level 5. Pure table lookup in `{vocab}`. Renumbered 5 → 6 because `to token` split into LM head + pick._"))

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@        PART 03 / STEP 6: decode — Level 6                @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

# ─── IN ──────────────────────────────────────────────────────
print("========== IN ==========")
print("new_token_id (tensor) :", new_token_id_tensor)
print("  shape               :", tuple(new_token_id_tensor.shape))
print("  dtype               :", new_token_id_tensor.dtype)
print("new_token_id (int)    :", new_token_id_int)
print("========================")
print()

# ─── PROCESS (transparent) ───────────────────────────────────
vocab_size_tok = tokenizer.vocab_size
vocab_size_cfg = config.vocab_size

print("========== PROCESS (transparent) ==========")
print("step                 : tokenizer.decode(new_token_id)")
print("op                   : inverse vocab lookup")
print("static used          : {vocab}  -- SAME BPE table used by tokenize in Step 1")
print("  tokenizer class    :", type(tokenizer).__name__)
print("  vocab_size (tok)   :", vocab_size_tok)
print("  vocab_size (cfg)   :", vocab_size_cfg)
print("  fast tokenizer     :", tokenizer.is_fast)
print("===========================================")
print()

token_str_raw = tokenizer.convert_ids_to_tokens(new_token_id_int)
text_piece    = tokenizer.decode([new_token_id_int])

print("========== INTERNAL VOCAB ROW (for inspection) ==========")
print("raw BPE token        :", repr(token_str_raw),
      "  <- 'Ġ' means leading space in GPT-style BPE")
print("decoded text piece   :", repr(text_piece))
print("=========================================================")
print()

# ─── OUT ──────────────────────────────────────────────────────
print("========== OUT ==========")
print("text_piece           :", repr(text_piece))
print("  type               :", type(text_piece).__name__)
print("  len (chars)        :", len(text_piece))
print("=========================")

# ─── Rich view ───────────────────────────────────────────────
display(Markdown("---"))
display(Markdown("### Rich view — Step 6"))

df_step6 = pd.DataFrame([
    {"side": "IN",      "name": "new_token_id",                       "value": str(new_token_id_int),
     "shape": str(tuple(new_token_id_tensor.shape)), "dtype": str(new_token_id_tensor.dtype),
     "ndim": new_token_id_tensor.ndim, "dim": "scalar (0-D)"},
    {"side": "PROCESS", "name": "tokenizer.decode (inverse vocab lookup)",
     "value": f"{type(tokenizer).__name__}, vocab_size={vocab_size_tok}",
     "shape": "—", "dtype": "—", "ndim": "—", "dim": "lookup (no math)"},
    {"side": "OUT",     "name": "text_piece",                         "value": repr(text_piece),
     "shape": "—", "dtype": "str", "ndim": "—", "dim": "string"},
]).set_index("side")
display(df_step6)

# ─── Final end-to-end summary (Level 6 pipeline: 6 steps) ────
display(Markdown("---"))
display(Markdown("### Level 6 pipeline — 6 steps"))

df_pipeline = pd.DataFrame([
    {"step": "tokenize", "op": "BPE segment + ID map",
     "in":  f"text  {repr(text)}",
     "static": "{vocab}",
     "out": f"token_ids  {tuple(token_ids_1d.shape)}  {token_ids_1d.dtype}",
     "dim_out": "1D vector"},
    {"step": "embed",    "op": "row lookup in {E}",
     "in":  f"token_ids  {tuple(token_ids_1d.shape)}  {token_ids_1d.dtype}",
     "static": f"{{E : {tuple(E.shape)}}}",
     "out": f"vectors  {tuple(vectors.shape)}  {vectors.dtype}",
     "dim_out": "2D matrix"},
    {"step": "LLM core", "op": "opaque (vectors -> vectors)",
     "in":  f"vectors  {tuple(vectors.shape)}  {vectors.dtype}",
     "static": "{core internals}",
     "out": f"vectors_out  {tuple(hidden_states.shape)}  {hidden_states.dtype}",
     "dim_out": "2D matrix"},
    {"step": "LM head",  "op": "matmul (vectors · {H})",
     "in":  f"vectors_out  {tuple(hidden_states.shape)}  {hidden_states.dtype}",
     "static": "{head internals}  ({H} opens at Level 7)",
     "out": f"logits  {tuple(logits.shape)}  {logits.dtype}",
     "dim_out": "2D matrix"},
    {"step": "pick",     "op": "argmax / sample",
     "in":  f"logits  {tuple(logits.shape)}  {logits.dtype}",
     "static": "—",
     "out": f"new_token_id  scalar  {new_token_id_tensor.dtype}",
     "dim_out": "scalar"},
    {"step": "decode",   "op": "inverse vocab lookup",
     "in":  f"new_token_id  scalar  {new_token_id_tensor.dtype}",
     "static": "{vocab}",
     "out": f"text_piece  {repr(text_piece)}",
     "dim_out": "string"},
])
df_pipeline.index = range(1, len(df_pipeline) + 1)
df_pipeline.index.name = "#"
display(df_pipeline)